In [1]:
import jsonlines

def get_file(feature, data_split):
    filename = f"mosaic_{data_split}_{feature}_translated.jsonl"
    data = None
    with jsonlines.open(filename) as reader:
        data = list(reader)
    return data

mosaic = {
    "test": {
        "instruction": get_file("instruction", "test"),
        "response": get_file("response", "test"),
    },
    "train": {
        "instruction": get_file("instruction", "train"),
        "response": get_file("response", "train"),
    },
}
    

In [3]:
from util import dataprep, map_and_index

# workaround to get the sentence mapping back on track
indexes = {}
for data_split in ["train", "test"]:
    instr_map, res_map = dataprep(data_split, prefix=">>nob<<")
    instr_index_map, _ = map_and_index(instr_map)
    res_index_map, _ = map_and_index(res_map)

    indexes[data_split] = {
        "instruction": instr_index_map,
        "response": res_index_map
    }

In [4]:
len(indexes["test"]["instruction"])

11596

In [5]:
indexes["test"]["instruction"][:10]

[(0, 0),
 (1, 1),
 (2, 2),
 (3, 2),
 (4, 2),
 (5, 3),
 (6, 3),
 (7, 4),
 (8, 5),
 (9, 5)]

In [6]:
for k, v in mosaic.items():
    for _k, _v in v.items():
        print(_k, len(_v))

instruction 11596
response 19919
instruction 115979
response 136151


In [7]:
from collections import defaultdict
import datasets
import pandas as pd

dataset = {}

for data_split in ["train", "test"]:
    tmp = {}
    for feature in ["instruction", "response"]:
        index = indexes[data_split][feature]
        data = mosaic[data_split][feature]
        grouped = defaultdict(list)
        translated_str = ""
        for (_, group_idx), sentence in zip(index, data):
            translated = sentence["translation"]
            grouped[group_idx].append(translated)
        # join all grouped as a single string
        tmp[feature] = grouped

    # join all grouped as a single string
    df = pd.DataFrame(tmp)
    df["instruction"] = df["instruction"].apply(lambda x: " ".join(x))
    df["response"] = df["response"].apply(lambda x: " ".join(x))
    dataset[data_split] = datasets.Dataset.from_pandas(df.reset_index(drop=True))

translated_mosaic = datasets.DatasetDict(dataset)
translated_mosaic

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 36751
    })
    test: Dataset({
        features: ['instruction', 'response'],
        num_rows: 5042
    })
})

In [72]:
translated_mosaic.push_to_hub("tollefj/nob-mosaikk-instruct")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/37 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]